# CSCI 4150 — Oct. 6 LLM Context Reverse-Engineering Notebook

**Purpose:** This notebook is a no-API, Colab-compatible starter activity for the Oct. 6 LLM lesson. It does **not** run a real LLM. Instead, it provides a transparent toy model that helps students see how token-like context can shift next-token predictions and how to record evidence.

**Core question:** How does an LLM turn context into generated language?

## Learning goals

By the end of this notebook, you should be able to:

1. Explain why changing context can change generated language.
2. Distinguish fluency from support/reliability.
3. Record what variable changed and what evidence was observed.
4. Prepare a project-specific LLM test plan for Oct. 9.

## Important limitation

This notebook uses a **toy next-token predictor**. It is intentionally simple and inspectable. A real LLM has learned representations from massive training data and uses complex neural architecture.

The point here is not to simulate a real model perfectly. The point is to make the experimental logic visible: **same task, different context, different output, different evidence.**

In [1]:
# Cell 1 — Setup
# This notebook uses only the Python standard library plus pandas if available.

import re
import math
import random
from collections import Counter, defaultdict

try:
    import pandas as pd
except Exception:
    pd = None

random.seed(4150)
print("Ready: no API key required.")

Ready: no API key required.


In [2]:
# Cell 2 — A tiny synthetic corpus
# The toy model uses these examples to estimate likely next words.
# Feel free to read this corpus. In a real LLM, the training data and learned weights are not visible this way.

CORPUS = """
A database systems course is useful for organizing data, schemas, SQL queries, and reliable storage.
A computer graphics course is useful for rendering images, visual simulations, animation, and interactive visual systems.
A statistical machine learning course is useful for predictive modeling, classification, regression, evaluation, and sensor data.
For a sensor failure project, statistical machine learning is often useful because it supports prediction from examples.
For a project with many records and existing SQL knowledge, database systems may help with storing and querying data.
When requirements are ambiguous, a reliable assistant should ask a clarifying question instead of inventing facts.
A confident answer can still be unsupported if the context does not contain the evidence.
"""

def tokenize(text):
    """Simple tokenization for teaching. Real tokenizers are more complex."""
    return re.findall(r"[A-Za-z0-9']+|[^\w\s]", text.lower())

tokens = tokenize(CORPUS)
print("First 50 tokens:")
print(tokens[:50])
print("Total tokens:", len(tokens))

First 50 tokens:
['a', 'database', 'systems', 'course', 'is', 'useful', 'for', 'organizing', 'data', ',', 'schemas', ',', 'sql', 'queries', ',', 'and', 'reliable', 'storage', '.', 'a', 'computer', 'graphics', 'course', 'is', 'useful', 'for', 'rendering', 'images', ',', 'visual', 'simulations', ',', 'animation', ',', 'and', 'interactive', 'visual', 'systems', '.', 'a', 'statistical', 'machine', 'learning', 'course', 'is', 'useful', 'for', 'predictive', 'modeling', ',']
Total tokens: 134


In [3]:
# Cell 3 — Build a transparent next-token table
# For each token, count which tokens have followed it in the tiny corpus.

next_counts = defaultdict(Counter)
for a, b in zip(tokens, tokens[1:]):
    next_counts[a][b] += 1

def next_token_distribution(context_text, top_k=8):
    """Return likely next tokens based on the last token in context."""
    ctx = tokenize(context_text)
    if not ctx:
        return []
    last = ctx[-1]
    counts = next_counts.get(last, Counter())
    total = sum(counts.values())
    if total == 0:
        return []
    return [(tok, count / total) for tok, count in counts.most_common(top_k)]

for context in ["statistical machine", "sensor", "unsupported"]:
    print("Context:", context)
    print(next_token_distribution(context))
    print()

Context: statistical machine
[('learning', 1.0)]

Context: sensor
[('data', 0.5), ('failure', 0.5)]

Context: unsupported
[('if', 1.0)]



## Part A — Context changes prediction

Run the next cell. Then change the `context_text` variable and run again.

Try examples like:

- `A statistical machine`
- `For a sensor failure project, statistical machine learning is`
- `When requirements are ambiguous, a reliable assistant should`

Record what changes.

In [4]:
 # Cell 4 — Inspect next-token predictions

context_text = "For a sensor failure project, statistical machine learning is"

dist = next_token_distribution(context_text, top_k=10)
print("Context:", context_text)
print("Likely next tokens:")
for tok, p in dist:
    print(f"  {tok!r:15s} {p:.2f}")

if not dist:
    print("No learned next-token continuation in this toy corpus.")

Context: For a sensor failure project, statistical machine learning is
Likely next tokens:
  'useful'        0.75
  'often'         0.25


In [5]:
# Cell 5 — A tiny generator
# This repeatedly predicts a next token using the transparent next-token table.

def generate_toy_text(prompt, steps=18, temperature=0.7):
    output = tokenize(prompt)
    for _ in range(steps):
        last = output[-1] if output else ""
        counts = next_counts.get(last, Counter())
        if not counts:
            break
        toks = list(counts.keys())
        weights = [counts[t] ** (1 / max(temperature, 0.01)) for t in toks]
        chosen = random.choices(toks, weights=weights, k=1)[0]
        output.append(chosen)
    return " ".join(output)

for prompt in [
    "a database systems course is",
    "a statistical machine learning course is",
    "when requirements are ambiguous"
]:
    print("PROMPT:", prompt)
    print(generate_toy_text(prompt, steps=16))
    print()

PROMPT: a database systems course is
a database systems course is useful because it supports prediction from examples . for organizing data . for a clarifying question

PROMPT: a statistical machine learning course is
a statistical machine learning course is useful for rendering images , and reliable assistant should ask a project , and existing sql

PROMPT: when requirements are ambiguous
when requirements are ambiguous , sql queries , a clarifying question instead of inventing facts . when requirements are ambiguous



## Part B — Controlled comparison

Below is a classroom-style experiment. The task stays similar, but the context changes.

For each condition, record:

- What variable changed?
- What output changed?
- Which claims are supported by the supplied context?
- Which claims are plausible but unsupported?
- What failure pattern appears?

In [10]:
# Cell 6 — Prompts for live LLM or toy-model comparison
# Use these with the approved conversational LLM in class, or adapt them for the toy generator.

conditions = {
    "Baseline": "Recommend one of Course A, Course B, or Course C for a student. Explain your reasoning.",
    "More context": "Course A is database systems. Course B is computer graphics. Course C is statistical machine learning. Recommend one for a student and explain your reasoning.",
    "Relevant evidence": "Course A is database systems. Course B is computer graphics. Course C is statistical machine learning. The student's project predicts equipment failure from sensor data, and the student already knows SQL. Recommend one course and explain your reasoning.",
    "Ambiguity": "Course A is database systems. Course B is computer graphics. Course C is statistical machine learning. The student's project predicts equipment failure from sensor data. The student cannot take a course requiring calculus, but you do not know which courses require calculus. Recommend one course and explain your reasoning."
}

for name, prompt in conditions.items():
    print("="*80)
    print(name.upper())
    print(prompt)
    print(generate_toy_text(prompt, steps=30))

BASELINE
Recommend one of Course A, Course B, or Course C for a student. Explain your reasoning.
recommend one of course a , course b , or course c for a student . explain your reasoning . when requirements are ambiguous , animation , regression , animation , visual systems course is useful for a computer graphics course is useful because it supports prediction from examples .
MORE CONTEXT
Course A is database systems. Course B is computer graphics. Course C is statistical machine learning. Recommend one for a student and explain your reasoning.
course a is database systems . course b is computer graphics . course c is statistical machine learning . recommend one for a student and explain your reasoning . a statistical machine learning is useful for a statistical machine learning course is often useful for organizing data , and reliable storage . for a reliable assistant should ask a
RELEVANT EVIDENCE
Course A is database systems. Course B is computer graphics. Course C is statistical 

In [23]:
# Cell 7 — Evidence table template
# After using the live LLM, paste concise summaries into this table.

rows = []
rows.append({
        "trial": "Baseline",
        "variable_changed": "none",
        "prediction_before_run": "The LLM will likely recommend a random course without having student background details.",
        "output_summary": "It suggested visual studio courses or random subjects unrelated to the target student.",
        "supported_claims": "None. All claims about student preferences were completely unsupported.",
        "unsupported_or_contradicted_claims": "The assertion that Course A is suitable for the student was unsupported.",
        "failure_pattern": "Hallucinates student attributes and course contents due to lack of context."
    })
rows.append({
        "trial": "More context",
        "variable_changed": "added course descriptions",
        "prediction_before_run": "The LLM will explain what the courses teach but still cannot match to the student's needs.",
        "output_summary": "It accurately described courses A, B, and C but guessed about student requirements.",
        "supported_claims": "Claims about course subjects are supported by the prompt.",
        "unsupported_or_contradicted_claims": "Claims about which course is best suited for the student are still unsupported.",
        "failure_pattern": "Fluent descriptions of courses, but the actual student match is completely speculative."
    })
rows.append({
        "trial": "Relevant evidence",
        "variable_changed": "added project details and prior SQL knowledge",
        "prediction_before_run": "The LLM will correctly select statistical machine learning (Course C) because of sensor failure predicting.",
        "output_summary": "It correctly recommended Course C for sensor data, and suggested Course A is redundant due to SQL knowledge.",
        "supported_claims": "Course C fits sensor failure modeling; student already knows SQL (no need for Course A database baseline).",
        "unsupported_or_contradicted_claims": "None, the reasoning is fully grounded in the provided prompt facts.",
        "failure_pattern": "None. Successful selection based on provided context matching."
    })
rows.append({
        "trial": "Ambiguity",
        "variable_changed": "added missing/unknown prerequisite constraint",
        "prediction_before_run": "The LLM will need to warn that it doesn't know which course requires calculus.",
        "output_summary": "It made a choice anyway, or failed to identify that it lacked critical prerequisite information.",
        "supported_claims": "The student has a sensor failure project (supported).",
        "unsupported_or_contradicted_claims": "The assertion that Course C does not require calculus (unsupported / assumed).",
        "failure_pattern": "Overconfident; fails to ask clarifying questions about calculus requirements, presenting assumptions as facts."
    })

if pd:
    evidence_table = pd.DataFrame(rows)
    display(evidence_table)
else:
    for row in rows:
        print(row)

,trial,variable_changed,prediction_before_run,output_summary,supported_claims,unsupported_or_contradicted_claims,failure_pattern
0,Baseline,none,The LLM will likely recommend a random course ...,It suggested visual studio courses or random s...,None. All claims about student preferences wer...,The assertion that Course A is suitable for th...,Hallucinates student attributes and course con...
1,More context,added course descriptions,The LLM will explain what the courses teach bu...,"It accurately described courses A, B, and C bu...",Claims about course subjects are supported by ...,Claims about which course is best suited for t...,"Fluent descriptions of courses, but the actual..."
2,Relevant evidence,added project details and prior SQL knowledge,The LLM will correctly select statistical mach...,It correctly recommended Course C for sensor d...,Course C fits sensor failure modeling; student...,"None, the reasoning is fully grounded in the p...",None. Successful selection based on provided c...
3,Ambiguity,added missing/unknown prerequisite constraint,The LLM will need to warn that it doesn't know...,"It made a choice anyway, or failed to identify...",The student has a sensor failure project (supp...,The assertion that Course C does not require c...,Overconfident; fails to ask clarifying questio...


### Fill-in version

Edit the Python strings below after your team runs the live LLM. This creates a saved summary table.

In [18]:
# Cell 8 — Team entries
# Replace the blank strings with your team's observations.

team_entries = [
    {
        "trial": "Baseline",
        "variable_changed": "none",
        "prediction_before_run": "The LLM will likely recommend a random course without having student background details.",
        "output_summary": "It suggested visual studio courses or random subjects unrelated to the target student.",
        "supported_claims": "None. All claims about student preferences were completely unsupported.",
        "unsupported_or_contradicted_claims": "The assertion that Course A is suitable for the student was unsupported.",
        "failure_pattern": "Hallucinates student attributes and course contents due to lack of context."
    },
    {
        "trial": "More context",
        "variable_changed": "added course descriptions",
        "prediction_before_run": "The LLM will explain what the courses teach but still cannot match to the student's needs.",
        "output_summary": "It accurately described courses A, B, and C but guessed about student requirements.",
        "supported_claims": "Claims about course subjects are supported by the prompt.",
        "unsupported_or_contradicted_claims": "Claims about which course is best suited for the student are still unsupported.",
        "failure_pattern": "Fluent descriptions of courses, but the actual student match is completely speculative."
    },
    {
        "trial": "Relevant evidence",
        "variable_changed": "added project details and prior SQL knowledge",
        "prediction_before_run": "The LLM will correctly select statistical machine learning (Course C) because of sensor failure predicting.",
        "output_summary": "It correctly recommended Course C for sensor data, and suggested Course A is redundant due to SQL knowledge.",
        "supported_claims": "Course C fits sensor failure modeling; student already knows SQL (no need for Course A database baseline).",
        "unsupported_or_contradicted_claims": "None, the reasoning is fully grounded in the provided prompt facts.",
        "failure_pattern": "None. Successful selection based on provided context matching."
    },
    {
        "trial": "Ambiguity",
        "variable_changed": "added missing/unknown prerequisite constraint",
        "prediction_before_run": "The LLM will need to warn that it doesn't know which course requires calculus.",
        "output_summary": "It made a choice anyway, or failed to identify that it lacked critical prerequisite information.",
        "supported_claims": "The student has a sensor failure project (supported).",
        "unsupported_or_contradicted_claims": "The assertion that Course C does not require calculus (unsupported / assumed).",
        "failure_pattern": "Overconfident; fails to ask clarifying questions about calculus requirements, presenting assumptions as facts."
    },
]

if pd:
    df = pd.DataFrame(team_entries)
    display(df)
else:
    for row in team_entries:
        print(row)

,trial,variable_changed,prediction_before_run,output_summary,supported_claims,unsupported_or_contradicted_claims,failure_pattern
0,Baseline,none,The LLM will likely recommend a random course ...,It suggested visual studio courses or random s...,None. All claims about student preferences wer...,The assertion that Course A is suitable for th...,Hallucinates student attributes and course con...
1,More context,added course descriptions,The LLM will explain what the courses teach bu...,"It accurately described courses A, B, and C bu...",Claims about course subjects are supported by ...,Claims about which course is best suited for t...,"Fluent descriptions of courses, but the actual..."
2,Relevant evidence,added project details and prior SQL knowledge,The LLM will correctly select statistical mach...,It correctly recommended Course C for sensor d...,Course C fits sensor failure modeling; student...,"None, the reasoning is fully grounded in the p...",None. Successful selection based on provided c...
3,Ambiguity,added missing/unknown prerequisite constraint,The LLM will need to warn that it doesn't know...,"It made a choice anyway, or failed to identify...",The student has a sensor failure project (supp...,The assertion that Course C does not require c...,Overconfident; fails to ask clarifying questio...


## Part C — Reliability labels

Use these labels consistently:

- **SUPPORTED:** Directly justified by supplied context.
- **PLAUSIBLE BUT UNSUPPORTED:** Sounds reasonable, but evidence is absent.
- **CONTRADICTED:** Conflicts with supplied context.
- **UNSTABLE:** Meaning changes across repeated or paraphrased prompts.
- **OVERCONFIDENT:** Uncertainty is present, but the response does not acknowledge it.

In [ ]:
# Cell 9 — Simple summary counts
# Edit these counts after your evidence table is complete.

summary_counts = {
    "useful_behavior_cases": 0,
    "unsupported_material_cases": 0,
    "contradicted_cases": 0,
    "unstable_cases": 0,
    "overconfident_cases": 0,
}

summary_counts

## Part D — Project experiment plan for Oct. 9

Do this only after the reverse-engineering activity. Your plan should test a **bounded project function**, not your whole project.

In [17]:
# Cell 10 — Project-specific LLM test plan
# Fill this in before leaving Oct. 6.

project_plan = {
    "project_name": "HI Medication Information and Interaction System",
    "bounded_llm_function": "extract medication details and identify potential drug interactions",
    "hypothesis": "We expect an LLM to help because it can parse unstructured medical texts and synthesize known drug contraindications effectively.",
    "manipulated_variable_for_oct9": "context amount and evidence constraint",
    "condition_A": "Only medication names provided (baseline interaction queries)",
    "condition_B": "Medication names provided with patient context, full ingredients list, and explicit interaction constraints",
    "normal_test_cases": [
        "Recommend interactions for Aspirin and Ibuprofen",
        "Summarize common side effects for Metformin",
        "Check if Acetaminophen conflicts with cold medicines"
    ],
    "hard_or_ambiguous_test_cases": [
        "Verify interactions for a complex compound prescription with missing patient allergy context",
        "Evaluate safety profile for a new experimental drug with no prior corpus history"
    ],
    "success_criteria": "The LLM correctly flags critical contraindications based on the provided drug guides without hallucinating unsupported reactions.",
    "failure_criteria": "The LLM fails to detect a high-risk medication interaction or claims a safe pairing is dangerous without evidence.",
    "data_safety_note": "We will use only public, synthetic, or authorized project data."
}

project_plan

{'project_name': 'HI Medication Information and Interaction System',
 'bounded_llm_function': 'extract medication details and identify potential drug interactions',
 'hypothesis': 'We expect an LLM to help because it can parse unstructured medical texts and synthesize known drug contraindications effectively.',
 'manipulated_variable_for_oct9': 'context amount and evidence constraint',
 'condition_A': 'Only medication names provided (baseline interaction queries)',
 'condition_B': 'Medication names provided with patient context, full ingredients list, and explicit interaction constraints',
 'normal_test_cases': ['Recommend interactions for Aspirin and Ibuprofen',
  'Summarize common side effects for Metformin',
  'Check if Acetaminophen conflicts with cold medicines'],
 'hard_or_ambiguous_test_cases': ['Verify interactions for a complex compound prescription with missing patient allergy context',
  'Evaluate safety profile for a new experimental drug with no prior corpus history'],
 's

In [21]:
# Cell 11 — Export a simple text plan to submit or paste into the dossier

from pathlib import Path

lines = ["CSCI 4150 Oct. 6 LLM Project Test Plan", "="*44, ""]
for key, value in project_plan.items():
    lines.append(f"{key}: {value}")

output_path = Path("oct6_llm_project_test_plan.txt")
output_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {output_path.resolve()}")
print("Copy this text into your Project Dossier or download the file from Colab.")

Wrote /content/oct6_llm_project_test_plan.txt
Copy this text into your Project Dossier or download the file from Colab.


## Final reflection

Answer before leaving:

1. What role did context play in generation?
2. Which variable did your team manipulate?
3. Where did fluency exceed evidence?
4. Where does meaning appear to live in this system?
5. What will your team test on Oct. 9?